In [ ]:
%%file producer.py
from kafka import KafkaProducer
import json
import random
import time
from datetime import datetime

# Konfiguracja producenta
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction(tx_counter):
    # Definicja list wartości losowych
    users = [f"u{i:02d}" for i in range(1, 21)]  # u01 do u20
    stores = ["Warszawa", "Kraków", "Gdańsk", "Wrocław"]
    categories = ["elektronika", "odzież", "żywność", "książki"]

    # Tworzenie obiektu transakcji
    transaction = {
        "tx_id": f"TX{tx_counter:04d}",
        "user_id": random.choice(users),
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(stores),
        "category": random.choice(categories),
        "timestamp": datetime.now().isoformat()
    }
    return transaction

print("Uruchamianie producenta transakcji... (Naciśnij Ctrl+C, aby przerwać)")

tx_id_counter = 1

try:
    while True:
        # Generowanie danych
        tx_data = generate_transaction(tx_id_counter)
        
        # Wysyłka do tematu 'transactions'
        producer.send('transactions', value=tx_data)
        
        # Wypisanie wysłanej transakcji w konsoli
        print(f"Wysłano: {tx_data}")
        
        # Inkrementacja licznika i pauza
        tx_id_counter += 1
        time.sleep(1)
        
except KeyboardInterrupt:
    print("\nZatrzymywanie producenta...")
finally:
    producer.flush()
    producer.close()

In [1]:
!python producer.py

NameError: name 'producer' is not defined

In [ ]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

# Konfiguracja konsumenta
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',  # Zaczyna czytać od początku, jeśli nie ma zapisanego offsetu
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    # Wyciągamy dane transakcji (słownik) z wartości wiadomości
    tx = message.value
    
    # TWÓJ KOD: sprawdzenie warunku amount > 3000
    if tx['amount'] > 3000:
        # Formatowanie komunikatu ALERT
        # ALERT: TX0042 | 3500.00 PLN | Warszawa | elektronika
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

In [ ]:
!python consumer_filter.py
producer.close()


In [ ]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

# Konfiguracja konsumenta
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrichment-group',  # Inny group_id niż w filtrze
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Uruchomiono wzbogacanie transakcji o poziom ryzyka...")

for message in consumer:
    tx = message.value
    amount = tx['amount']
    
    # Logika wzbogacania danych (Risk Level)
    if amount > 3000:
        risk_level = "HIGH"
    elif amount > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
    
    # Dodanie nowego pola do słownika
    tx['risk_level'] = risk_level
    
    # Wypisanie wzbogaconego eventu
    print(f"[{risk_level}] ID: {tx['tx_id']} | Kwota: {amount} | Sklep: {tx['store']}")

In [ ]:
!python consumer_enrich.py


In [ ]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

# Konfiguracja konsumenta
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Uruchomiono moduł statystyk sprzedaży...")

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    
    # 1. Aktualizacja statystyk
    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0.0) + amount
    msg_count += 1
    
    # 2. Co 10 wiadomości wypisz tabelę
    if msg_count % 10 == 0:
        print("\n" + "="*55)
        print(f"{'SKLEP':<12} | {'LICZBA':<8} | {'SUMA':<12} | {'ŚREDNIA':<10}")
        print("-" * 55)
        
        for s in sorted(store_counts.keys()):
            count = store_counts[s]
            suma = total_amount[s]
            srednia = suma / count
            print(f"{s:<12} | {count:<8} | {suma:>9.2f} zł | {srednia:>8.2f} zł")
        
        print("="*55)


In [ ]:
!python consumer_count.py

In [ ]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

# Konfiguracja konsumenta
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='category-stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Inicjalizacja struktur danych
category_stats = {}
msg_count = 0

print("Analiza statystyk per kategoria uruchomiona...")

for message in consumer:
    tx = message.value
    cat = tx['category']
    amt = tx['amount']
    
    # Inicjalizacja kategorii, jeśli pojawia się pierwszy raz
    if cat not in category_stats:
        category_stats[cat] = {
            'count': 0,
            'total': 0.0,
            'min': amt,
            'max': amt
        }
    
    # Aktualizacja danych
    stats = category_stats[cat]
    stats['count'] += 1
    stats['total'] += amt
    stats['min'] = min(stats['min'], amt)
    stats['max'] = max(stats['max'], amt)
    
    msg_count += 1
    
    # Raport co 10 wiadomości
    if msg_count % 10 == 0:
        print(f"\n--- RAPORT KATEGORII (Wiadomość #{msg_count}) ---")
        print(f"{'KATEGORIA':<12} | {'SZTUK':<5} | {'SUMA':>10} | {'MIN':>8} | {'MAX':>8}")
        print("-" * 55)
        
        for c, s in sorted(category_stats.items()):
            print(f"{c:<12} | {s['count']:<5} | {s['total']:>10.2f} | {s['min']:>8.2f} | {s['max']:>8.2f}")
        print("-" * 55)

In [ ]:
!python consumer_stats.py

In [ ]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime

# Konfiguracja konsumenta
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='latest',  # Wykrywamy anomalie w czasie rzeczywistym
    group_id='anomaly-detector-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Słownik przechowujący listy timestampów dla każdego użytkownika
# user_history[user_id] = [timestamp1, timestamp2, ...]
user_history = defaultdict(list)

print("System wykrywania anomalii uruchomiony (limit: 3 transakcje / 60s)...")

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    # Konwersja ISO timestamp ze stringa na obiekt datetime
    current_time = datetime.fromisoformat(tx['timestamp'])
    
    # 1. Dodaj obecną transakcję do historii użytkownika
    user_history[user_id].append(current_time)
    
    # 2. Usuń z historii wpisy starsze niż 60 sekund względem obecnej transakcji
    # Dzięki temu sprawdzamy tylko "ruchome okno" czasu
    user_history[user_id] = [
        t for t in user_history[user_id] 
        if (current_time - t).total_seconds() <= 60
    ]
    
    # 3. Sprawdź warunek anomalii
    if len(user_history[user_id]) > 3:
        print(f"ALARM ANOMALII: Użytkownik {user_id} wykonał "
              f"{len(user_history[user_id])} transakcji w ciągu ostatnich 60 sekund!")
        print(f"   Ostatnia transakcja: {tx['tx_id']} | Sklep: {tx['store']}")

In [ ]:
!python consumer_anomaly.py